# Clean pipeline, step by step

Runs `clean_data.STEPS` one at a time and keeps the frames after every step, so any
intermediate table can be inspected. Nothing is written to disk unless the last cell is run.


In [1]:
%load_ext autoreload
%autoreload 2

import time

import polars as pl

from money_printer_research.category_theory import Err, Ok
from money_printer_research.clean_data import STEPS, save_frames
from money_printer_research.config import settings

pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(40)

polars.config.Config

## Run

In [2]:
def run(steps, start):
    """Run steps in order; keep each step's output. Stops at the first Err."""
    snapshots = {}
    value = start
    for step in steps:
        t0 = time.perf_counter()
        result = step(value)
        took = time.perf_counter() - t0
        match result:
            case Ok(frames):
                snapshots[step.__name__] = frames
                value = frames
                print(f"ok    {took:6.1f}s  {step.__name__}")
            case Err(reason, cause):
                print(f"FAIL  {took:6.1f}s  {step.__name__}\n\n{reason}")
                return snapshots, result
    return snapshots, Ok(value)


snapshots, result = run(STEPS, settings)

ok       0.4s  _read_all
ok       0.0s  functor(snake_columns_pl)
ok       0.1s  _check_aliases
ok       0.0s  functor(rename_team_names)
ok       0.1s  _add_season_start
ok       0.0s  _check_unplayed
ok       0.0s  _split_fixtures
ok       0.0s  _validate('xg', 'fbref', 'player_stat')
ok       0.0s  _build_matches
ok       0.1s  _add_elo
ok       0.1s  _build_player_matches
ok       7.8s  _add_player_feat
ok       0.6s  _add_team_calendar
ok       0.0s  _validate('player_matches', 'matches')


## Frame sizes after each step

In [3]:
pl.DataFrame(
    [
        {"step": step, "frame": name, "rows": df.height, "cols": df.width}
        for step, frames in snapshots.items()
        for name, df in frames.items()
    ]
).pivot(on="frame", index="step", values="rows")

step,fbref,xg,player_stat,tm_games,tm_competitions,fixtures,matches,player_matches
str,i64,i64,i64,i64,i64,i64,i64,i64
"""_read_all""",144106,21690,630291,28140,65,null,null,null
"""functor(snake_columns_pl)""",144106,21690,630291,28140,65,null,null,null
"""_check_aliases""",144106,21690,630291,28140,65,null,null,null
"""functor(rename_team_names)""",144106,21690,630291,28140,65,null,null,null
"""_add_season_start""",144106,21690,630291,28140,65,null,null,null
…,…,…,…,…,…,…,…,…
"""_add_elo""",144106,21589,630291,28140,65,101,21589,null
"""_build_player_matches""",144106,21589,630291,28140,65,101,21589,630291
"""_add_player_feat""",144106,21589,630291,28140,65,101,21589,630291


## Inspect the final frames

In [4]:
frames = result.value if isinstance(result, Ok) else list(snapshots.values())[-1]
matches = frames["matches"]
matches.head()

league,season,season_start,game_id,match_date,home_team,away_team,full_time_home_goals,full_time_away_goals,full_time_result,…,h_prev_europe_away,h_days_to_next_all,h_next_is_europe,a_rest_days_all,a_games_7d_all,a_games_14d_all,a_non_league_14d,a_prev_europe_away,a_days_to_next_all,a_next_is_europe
str,str,i64,i64,date,str,str,i64,i64,str,…,bool,i64,bool,i64,i64,i64,i64,bool,i64,bool
"""FRA-Ligue 1""","""2014/15""",2014,6185,2014-08-08,"""reims""","""paris saint germain""",2,2,"""D""",…,false,9,false,6,1,1,1,false,8,false
"""FRA-Ligue 1""","""2014/15""",2014,6186,2014-08-09,"""sc bastia""","""marseille""",3,3,"""D""",…,false,7,false,null,0,0,0,false,8,false
"""FRA-Ligue 1""","""2014/15""",2014,6187,2014-08-09,"""evian thonon gaillard""","""caen""",0,3,"""A""",…,false,7,false,null,0,0,0,false,6,false
"""FRA-Ligue 1""","""2014/15""",2014,6188,2014-08-09,"""guingamp""","""saint-etienne""",0,2,"""A""",…,false,7,false,null,0,0,0,false,8,false
"""FRA-Ligue 1""","""2014/15""",2014,6189,2014-08-09,"""lille""","""metz""",0,0,"""D""",…,false,6,false,null,0,0,0,false,7,false


In [5]:
# Columns with nulls, as a share of rows
nulls = matches.null_count().unpivot(variable_name="column", value_name="nulls")
nulls.filter(pl.col("nulls") > 0).with_columns(
    (pl.col("nulls") / matches.height).round(4).alias("share")
).sort("nulls", descending=True)

column,nulls,share
str,u32,f64
"""a_days_to_next_all""",64,0.003
"""h_days_to_next_all""",58,0.0027
"""a_rest_days_all""",41,0.0019
"""h_rest_days_all""",35,0.0016
"""a_player11_xg90""",8,0.0004
…,…,…
"""h_missing_q""",2,0.0001
"""a_missing_q""",2,0.0001
"""h_missing_regulars""",2,0.0001


In [6]:
# Games per league and season
matches.pivot(on="league", index="season", values="game_id", aggregate_function="len").sort("season")

season,FRA-Ligue 1,ENG-Premier League,GER-Bundesliga,ESP-La Liga,ITA-Serie A
str,u32,u32,u32,u32,u32
"""2014/15""",380,380,306,380,380
"""2015/16""",380,380,306,380,380
"""2016/17""",380,380,306,380,380
"""2017/18""",380,380,306,380,380
"""2018/19""",380,380,306,380,380
…,…,…,…,…,…
"""2021/22""",380,380,306,380,380
"""2022/23""",380,380,306,380,380
"""2023/24""",306,380,306,380,380


In [7]:
# Any intermediate table, e.g. the xg frame right after the alias check
snapshots["_check_aliases"]["xg"].head()

league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
str,i64,str,i64,i64,i64,datetime[μs],i64,i64,str,str,str,str,i64,i64,f64,f64,bool,bool,str
"""ENG-Premier League""",1415,"""2014-08-16 Arsenal-Crystal Palace""",1,2014,4755,2014-08-16 17:30:00,83,78,"""Arsenal""","""Crystal Palace""","""CRY""","""ARS""",2,1,1.55411,0.158151,true,true,"""https://understat.com/match/4755"""
"""ENG-Premier League""",1415,"""2014-08-16 Leicester-Everton""",1,2014,4750,2014-08-16 15:00:00,75,72,"""Leicester""","""Everton""","""EVE""","""LEI""",2,2,1.2783,0.613273,true,true,"""https://understat.com/match/4750"""
"""ENG-Premier League""",1415,"""2014-08-16 Manchester United-Swansea""",1,2014,4749,2014-08-16 12:45:00,89,84,"""Manchester United""","""Swansea""","""SWA""","""MUN""",1,2,1.16635,0.278076,true,true,"""https://understat.com/match/4749"""
"""ENG-Premier League""",1415,"""2014-08-16 Queens Park Rangers-Hull""",1,2014,4751,2014-08-16 15:00:00,202,91,"""Queens Park Rangers""","""Hull""","""HUL""","""QPR""",0,1,1.90067,1.11757,true,true,"""https://understat.com/match/4751"""
"""ENG-Premier League""",1415,"""2014-08-16 Stoke-Aston Villa""",1,2014,4752,2014-08-16 15:00:00,85,71,"""Stoke""","""Aston Villa""","""AVL""","""STO""",0,1,0.423368,0.909774,true,true,"""https://understat.com/match/4752"""


## Save (optional)

Same as `uv run clean_data`: writes every frame to `data/clean/`.

In [8]:
SAVE = False

if SAVE and isinstance(result, Ok):
    for path in save_frames(result.value, settings.clean.out_dir):
        print("saved", path)